# Notebook 02 — Thu thập và Làm sạch

Pipeline xử lý cho dữ liệu căn hộ/chung cư TP.HCM.

In [1]:
import pandas as pd
from src.data_manager import PropertyDataManager

mgr = PropertyDataManager('data/raw/real_estate_apartment.xlsx')
raw = mgr.load_raw()
print('Trư�c làm sạch:')
print(f'  Số dòng: {len(raw)}')
print(f'  Số cột: {raw.shape[1]}')
print(f'  district unique: {raw["district"].nunique()} giá trị — {raw["district"].unique().tolist()}')
print(f'  direction range: {raw["direction"].min():.0f}-{raw["direction"].max():.0f} (mã hướng 1..8)')
print(f'  area_m2: min={raw["area_m2"].min()}, max={raw["area_m2"].max()}')

Trư�c làm sạch:
  Số dòng: 1799
  Số cột: 32
  district unique: 6 giá trị — ['Quận Bình Thạnh', 'Quận Gò Vấp', 'Quận 12', 'Thành phố Thủ Đức', 'Quận 7', 'Quận Bình Tân']
  direction range: 1-8 (mã hướng 1..8)
  area_m2: min=24.0, max=1323.0


## 1. Chuẩn hoá district

Data xlsx đã có district dạng text chuẩn — `normalize_district` chỉ pass-through + strip.

In [2]:
from src.cleaner import normalize_district
raw['district_clean'] = raw['district'].apply(normalize_district)
print('Trước — 10 giá trị district đầu:')
print(raw['district'].head(10).tolist())
print('\nSau — 10 district_clean đầu:')
print(raw['district_clean'].head(10).tolist())

Trước — 10 giá trị district đầu:
['Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh']

Sau — 10 district_clean đầu:
['Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh', 'Quận Bình Thạnh']


## 2. Decode direction code (1..8) → tên hướng chính

In [3]:
from src.cleaner import normalize_direction
raw['direction_clean'] = raw['direction'].apply(normalize_direction)
print('Phân bố direction_clean:')
print(raw['direction_clean'].value_counts(dropna=False))

Phân bố direction_clean:
direction_clean
NaN         1420
Tây Nam       79
Nam           49
Đông Bắc      48
Bắc           45
Tây           42
Tây Bắc       40
Đông          39
Đông Nam      37
Name: count, dtype: int64


## 3. Chạy pipeline làm sạch

Quy tắc lọc outlier cho căn hộ:
- `area_m2 < 10` hoặc `> 500` → drop
- `total_price < 100_000_000` → drop
- `bedrooms > 10` → drop

In [4]:
cleaned, log, errors = mgr.clean()
print(f'Sau làm sạch: {len(cleaned)} dòng (bỏ {len(raw) - len(cleaned)} outlier)')
print(f'Cleaning log: {len(log)} dòng được ghi nhận')
print(f'\nPhân bố issue_type trong log:')
print(log['issue_type'].value_counts())
print(f'\nErrors: {errors}')

Sau làm sạch: 1779 dòng (bỏ 20 outlier)
Cleaning log: 20 dòng được ghi nhận

Phân bố issue_type trong log:
issue_type
bedrooms_outlier     17
area_out_of_range     3
Name: count, dtype: int64

Errors: []


## 4. Merge tiện ích (amenities) theo (district_clean, ward)

In [5]:
amenities = pd.read_csv('data/raw/neighborhood_amenities.csv')
merged = mgr.merge_amenities(cleaned, amenities)
print(f'Merged shape: {merged.shape}')
print(f'Tin khớp amenity: {merged["amenity_score"].notna().sum()} / {len(merged)}')

Merged shape: (1779, 41)
Tin khớp amenity: 1763 / 1779


## 5. Lưu kết quả

In [6]:
from pathlib import Path
mgr.save_cleaned(
    cleaned, log,
    cleaned_path=Path('data/processed/listings_clean.csv'),
    log_path=Path('data/logs/cleaning_log.csv'),
    error_path=Path('data/logs/error_log.txt'),
    errors=errors,
)
merged.to_csv('data/processed/listings_with_amenities.csv', index=False)
print('Saved → data/processed/ và data/logs/')

Saved → data/processed/ và data/logs/


## Tổng kết

- Raw: **1799 dòng × 32 cột** (căn hộ tại 6 quận TP.HCM)
- Cleaned: **1779 dòng × 40 cột** (drop 20 outliers — 17 bedrooms > 10, 3 area ngoài range)
- Merged with amenities: **1779 dòng × 41 cột** (99% coverage)
- 5 cột mới thêm: `district_clean`, `direction_clean`, `furnishing_label`, `legal_label`, và các cột mã (direction_code/balcony_code/furnishing_code/legal_code/apartment_type)